In [2]:
import requests
import os

In [3]:
MGNIFY_API_BASE = "https://www.ebi.ac.uk/metagenomics/api/v1"
OUTPUT_DIR = "/dataset/mgnify"


In [4]:
def get_human_gut_studies():
    """Fetch studies related to human gut from MGnify API."""
    url = f"{MGNIFY_API_BASE}/biomes/root:Host-associated:Human:Digestive system:Large intestine/studies"
    # params = {id:"root:Host-associated:Human:Digestive%20system:Large%20intestine", "page_size": 300}
    response = requests.get(url)
    response.raise_for_status()
    studies = response.json()["data"]
    return [study["id"] for study in studies]

def get_study_samples(study_id):
    """Retrieve samples for a given study."""
    url = f"{MGNIFY_API_BASE}/studies/{study_id}/samples"
    response = requests.get(url)
    response.raise_for_status()
    return response.json()["data"]

def get_sample_runs(sample_id):
    """Get sequencing runs for a given sample."""
    url = f"{MGNIFY_API_BASE}/samples/{sample_id}/runs"
    response = requests.get(url)
    response.raise_for_status()
    return response.json()["data"]

def download_file(url, filename):
    """Download a file from a given URL."""
    response = requests.get(url, stream=True)
    response.raise_for_status()
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    with open(filename, "wb") as file:
        for chunk in response.iter_content(chunk_size=1024):
            file.write(chunk)
    print(f"Downloaded: {filename}")


In [5]:
print("Fetching human gut microbiome studies from MGnify...")
study_ids = get_human_gut_studies()

for study_id in study_ids:
    print(f"Processing study: {study_id}")
    samples = get_study_samples(study_id)

    for sample in samples:
        sample_id = sample["id"]
        print(f"  - Sample: {sample_id}")

        runs = get_sample_runs(sample_id)
        for run in runs:
            fastq_url = run["attributes"].get("fastq_files", [None])[0]
            if fastq_url:
                filename = os.path.join(OUTPUT_DIR, study_id, sample_id, os.path.basename(fastq_url))
                download_file(fastq_url, filename)

Fetching human gut microbiome studies from MGnify...
Processing study: MGYS00005333
  - Sample: ERS1015689
  - Sample: ERS1015679
  - Sample: ERS1015687
  - Sample: ERS1015662
  - Sample: ERS1015779
  - Sample: ERS1015861
  - Sample: ERS1015844
  - Sample: ERS1015854
  - Sample: ERS1015683


KeyboardInterrupt: 

In [6]:
study_ids = get_human_gut_studies()
study_ids

['MGYS00005333',
 'MGYS00005230',
 'MGYS00002061',
 'MGYS00000633',
 'MGYS00005154',
 'MGYS00006551',
 'MGYS00005023',
 'MGYS00002425',
 'MGYS00003468',
 'MGYS00005234',
 'MGYS00003511',
 'MGYS00001189',
 'MGYS00001255',
 'MGYS00003575',
 'MGYS00006553',
 'MGYS00001278',
 'MGYS00000580',
 'MGYS00001211',
 'MGYS00006023',
 'MGYS00006121',
 'MGYS00006226',
 'MGYS00006224',
 'MGYS00006559',
 'MGYS00006431',
 'MGYS00006453']

In [16]:
import json
metadata_dict = {}
secondary_accessions = []
for study_id in study_ids:
    print(f"Processing study: {study_id}")
    study_link = f"{MGNIFY_API_BASE}/studies/{study_id}"
    response = requests.get(study_link)
    response.raise_for_status()
    data = response.json()['data']
    secondary_accessions.append(data['attributes']['secondary-accession'])
    metadata_dict[study_id] = data

# save to metadata json
filename = "/dataset/mgnify/metadata.json"
with open(filename, "w") as file:
    json.dump(metadata_dict, file)

Processing study: MGYS00005333
Processing study: MGYS00005230
Processing study: MGYS00002061
Processing study: MGYS00000633
Processing study: MGYS00005154
Processing study: MGYS00006551
Processing study: MGYS00005023
Processing study: MGYS00002425
Processing study: MGYS00003468
Processing study: MGYS00005234
Processing study: MGYS00003511
Processing study: MGYS00001189
Processing study: MGYS00001255
Processing study: MGYS00003575
Processing study: MGYS00006553
Processing study: MGYS00001278
Processing study: MGYS00000580
Processing study: MGYS00001211
Processing study: MGYS00006023
Processing study: MGYS00006121
Processing study: MGYS00006226
Processing study: MGYS00006224
Processing study: MGYS00006559
Processing study: MGYS00006431
Processing study: MGYS00006453


In [19]:
for i, study_id in enumerate(study_ids):
    print(f"Processing study: {study_id}")
    acc = secondary_accessions[i]
    download_link = f"{MGNIFY_API_BASE}/studies/{study_id}/pipelines/5.0/file/{acc}_taxonomy_abundances_SSU_v5.0.tsv"
    response = requests.get(download_link, stream=True)
    response.raise_for_status()
    filename = f"/home/kevin/Desktop/gut_microbiome/dataset/mgnify/taxonomy_abundances/{study_id}.tsv"
    with open(filename, "wb") as file:
        for chunk in response.iter_content(chunk_size=1024):
            file.write(chunk)
    print(f"Downloaded: {filename}")

Processing study: MGYS00005333
Downloaded: /home/kevin/Desktop/gut_microbiome/dataset/mgnify/taxonomy_abundances/MGYS00005333.tsv
Processing study: MGYS00005230
Downloaded: /home/kevin/Desktop/gut_microbiome/dataset/mgnify/taxonomy_abundances/MGYS00005230.tsv
Processing study: MGYS00002061
Downloaded: /home/kevin/Desktop/gut_microbiome/dataset/mgnify/taxonomy_abundances/MGYS00002061.tsv
Processing study: MGYS00000633
Downloaded: /home/kevin/Desktop/gut_microbiome/dataset/mgnify/taxonomy_abundances/MGYS00000633.tsv
Processing study: MGYS00005154
Downloaded: /home/kevin/Desktop/gut_microbiome/dataset/mgnify/taxonomy_abundances/MGYS00005154.tsv
Processing study: MGYS00006551
Downloaded: /home/kevin/Desktop/gut_microbiome/dataset/mgnify/taxonomy_abundances/MGYS00006551.tsv
Processing study: MGYS00005023
Downloaded: /home/kevin/Desktop/gut_microbiome/dataset/mgnify/taxonomy_abundances/MGYS00005023.tsv
Processing study: MGYS00002425
Downloaded: /home/kevin/Desktop/gut_microbiome/dataset/mgni

In [8]:
for id in secondary_accessions:
    download_link = f"https://www.ebi.ac.uk/metagenomics/api/v1/studies/{id}/downloads"
    response = requests.get(download_link, stream=True)
    response.raise_for_status()
    data = response.json()["data"]
    print(data)
    break

[{'type': 'study-downloads', 'id': 'ERP108859_GO-slim_abundances_v4.1.tsv', 'attributes': {'alias': 'ERP108859_GO-slim_abundances_v4.1.tsv', 'file-format': {'name': 'TSV', 'extension': 'tsv', 'compression': False}, 'description': {'label': 'GO slim annotation', 'description': 'GO slim annotation'}, 'group-type': 'Functional analysis', 'file-checksum': {'checksum': '', 'checksum-algorithm': ''}}, 'relationships': {'pipeline': {'data': {'type': 'pipelines', 'id': '4.1'}, 'links': {'related': 'https://www.ebi.ac.uk/metagenomics/api/v1/pipelines/4.1'}}}, 'links': {'self': 'https://www.ebi.ac.uk/metagenomics/api/v1/studies/MGYS00005333/pipelines/4.1/file/ERP108859_GO-slim_abundances_v4.1.tsv'}}, {'type': 'study-downloads', 'id': 'ERP108859_GO_abundances_v4.1.tsv', 'attributes': {'alias': 'ERP108859_GO_abundances_v4.1.tsv', 'file-format': {'name': 'TSV', 'extension': 'tsv', 'compression': False}, 'description': {'label': 'Complete GO annotation', 'description': 'Complete GO annotation'}, 'gr

In [13]:
data[3]

{'type': 'study-downloads',
 'id': 'ERP108859_phylum_taxonomy_abundances_SSU_v4.1.tsv',
 'attributes': {'alias': 'ERP108859_phylum_taxonomy_abundances_SSU_v4.1.tsv',
  'file-format': {'name': 'TSV', 'extension': 'tsv', 'compression': False},
  'description': {'label': 'Phylum level taxonomies SSU',
   'description': 'Phylum level taxonomies SSU (TSV)'},
  'group-type': 'Taxonomic analysis SSU rRNA',
  'file-checksum': {'checksum': '', 'checksum-algorithm': ''}},
 'relationships': {'pipeline': {'data': {'type': 'pipelines', 'id': '4.1'},
   'links': {'related': 'https://www.ebi.ac.uk/metagenomics/api/v1/pipelines/4.1'}}},
 'links': {'self': 'https://www.ebi.ac.uk/metagenomics/api/v1/studies/MGYS00005333/pipelines/4.1/file/ERP108859_phylum_taxonomy_abundances_SSU_v4.1.tsv'}}

In [3]:
download_link = "https://www.ebi.ac.uk/metagenomics/api/v1/studies/MGYS00000258/pipelines/1.0/file/SRP000319_taxonomy_abundances_v1.0.tsv"
response = requests.get(download_link, stream=True)
response.raise_for_status()
filename = "/dataset/mgnify/text.tsv"
with open(filename, "wb") as file:
    for chunk in response.iter_content(chunk_size=1024):
        file.write(chunk)
print(f"Downloaded: {filename}")

Downloaded: /home/kevin/Desktop/gut_microbiome/dataset/mgnify/text.tsv
